<a href="https://colab.research.google.com/github/llelus/DSA-Project/blob/main/notebooks/04_ml_shap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Machine learning: is BTC information predictive out of sample?

Target: Polymarket price change over the next 5 minutes, `y = price_{t+5} − price_t`.
Three LightGBM regressors with identical settings:

| Model | Features (all known at t) |
|---|---|
| Poly-only | price, last 5 one-minute changes, staleness, time to expiry |
| BTC-only | fair value, gap, moneyness, volatility, last 5 fair-value changes |
| Full | both |

Under semi-strong efficiency, adding BTC information should **not** improve out-of-sample forecasts.
Metrics: R² against the "no change" forecast (random-walk benchmark) and directional AUC on rows where
the price actually moves. Folds are blocks of whole expiry hours in time order; early stopping uses the
end of the training block; CIs come from a bootstrap over expiry hours.

*(v1 predicted "next tick up?" on data where most prices sat at 0/1 after expiry, so its AUC of 0.97 mostly
measured whether a market had already resolved.)*

In [ ]:
# Setup — identical in every notebook; works in Colab and locally
import os, sys, subprocess
BRANCH = "main"   # GitHub branch Colab runs; it must contain src/
if "google.colab" in sys.modules:
    REPO = "/content/DSA-Project"
    URL = "https://github.com/llelus/DSA-Project.git"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, URL, REPO], check=True)
    else:  # runtime reused: make sure the requested branch is checked out and current
        subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO, "pull", "-q", "--autostash", "origin", BRANCH], check=True)
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if not os.path.isdir("src"):
    raise RuntimeError(f"src/ not found in {os.getcwd()}: branch '{BRANCH}' on GitHub does not "
                       "contain the v2 code yet (push/merge it, or set BRANCH).")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.metrics import roc_auc_score
from src import config, dataset, analysis, ml
from src.results import save_metrics, savefig

panel = dataset.add_market_diffs(dataset.load_panel())
usable = panel["fair"].notna() & (panel["tau_min"] >= config.MIN_TAU_MIN)
mid = usable & panel["price"].between(config.MID_LOW, config.MID_HIGH)
print(f"Panel: {len(panel):,} market-minutes | {panel.condition_id.nunique():,} markets | "
      f"{panel.market_end.nunique():,} expiry hours")
print(f"Usable rows: {usable.sum():,} | in the {config.MID_LOW}-{config.MID_HIGH} price band: {mid.sum():,}")

data = ml.make_dataset(panel, mask=mid)
print(f"ML rows: {len(data):,} over {data.market_end.nunique():,} expiry hours; "
      f"share with y != 0: {(data.y != 0).mean():.1%}")

In [ ]:
preds, iters = {}, {}
for name, feats in [("poly_only", ml.POLY_FEATURES), ("btc_only", ml.BTC_FEATURES), ("full", ml.FULL_FEATURES)]:
    preds[name], iters[name] = ml.cv_predict(data, feats)
    print(f"{name:9s} best iterations per fold: {iters[name]}")

y = data.y.to_numpy()
oof = ~np.isnan(preds["full"])
moving = oof & (y != 0)
table = pd.DataFrame({name: {
    "r2_vs_no_change": ml.r2_vs_zero(y[oof], p[oof]),
    "directional_auc": roc_auc_score(y[moving] > 0, p[moving])} for name, p in preds.items()}).T
print(table.round(4))

In [ ]:
clusters = data.market_end.to_numpy()
r2_ci = analysis.cluster_bootstrap_diff(ml.r2_vs_zero, preds["full"][oof], preds["poly_only"][oof],
                                        y[oof], clusters[oof], n_boot=config.N_BOOT)
auc_ci = analysis.cluster_bootstrap_diff(lambda yy, pp: roc_auc_score(yy > 0, pp),
                                         preds["full"][moving], preds["poly_only"][moving],
                                         y[moving], clusters[moving], n_boot=config.N_BOOT)
print(f"Full − Poly-only R²:  {r2_ci['diff']:+.4f}  95% CI [{r2_ci['lo']:+.4f}, {r2_ci['hi']:+.4f}]")
print(f"Full − Poly-only AUC: {auc_ci['diff']:+.4f}  95% CI [{auc_ci['lo']:+.4f}, {auc_ci['hi']:+.4f}]")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
table["r2_vs_no_change"].plot.bar(ax=axes[0], color="#378ADD"); axes[0].axhline(0, color="k", lw=.8)
axes[0].set_title("Out-of-sample R² vs 'no change'")
table["directional_auc"].plot.bar(ax=axes[1], color="#D85A30"); axes[1].axhline(.5, color="k", lw=.8)
axes[1].set_ylim(.4, max(.75, table.directional_auc.max() + .05)); axes[1].set_title("Directional AUC (moving rows)")
for ax in axes: ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); savefig(fig, "ml_model_comparison"); plt.show()
save_metrics("ml", horizon_min=config.ML_HORIZON_MIN, n_rows=len(data), n_oof=int(oof.sum()),
             table=table.to_dict(orient="index"), full_minus_poly_r2=r2_ci, full_minus_poly_auc=auc_ci)

## SHAP — which information drives the forecast?

Final model trained on all rows with the median best-iteration count from the CV folds;
SHAP values on a random sample of 20,000 rows.

In [ ]:
final = ml.fit_final(data, ml.FULL_FEATURES, np.median(iters["full"]))
sample = data.sample(min(20_000, len(data)), random_state=config.RANDOM_STATE)[ml.FULL_FEATURES]
sv = shap.TreeExplainer(final)(sample)

shap.plots.beeswarm(sv, max_display=12, show=False)
plt.title("SHAP — full model"); savefig(plt.gcf(), "ml_shap_beeswarm"); plt.show()

imp = pd.Series(np.abs(sv.values).mean(0), index=ml.FULL_FEATURES).sort_values(ascending=False)
btc_share = imp[ml.BTC_FEATURES].sum() / imp.sum()
print(imp.round(5)); print(f"\nShare of mean |SHAP| from BTC features: {btc_share:.1%}")
save_metrics("ml_shap", mean_abs_shap=imp.to_dict(), btc_share=float(btc_share))

## Reading the results

* If **Full > Poly-only** with a CI excluding 0, BTC information public at t predicts Polymarket's next
  moves → it is not instantly priced (consistent with a lag in notebook 03).
* If the difference is ~0, Polymarket already reflects BTC information at the 5-minute horizon.
* The SHAP share of BTC features shows how much of the forecastable movement is "catching up with BTC"
  versus Polymarket's own dynamics.

In [ ]:
# Optional: commit outputs back to GitHub (needs Colab secrets GITHUB_TOKEN, GIT_NAME, GIT_EMAIL)
PUSH = False
if PUSH:
    from src.colab import push
    push("results: notebook 04 (ML + SHAP)", paths=('figures', 'results'))